# AnyUp 3D Training Notebook

This notebook trains the 3D `AnyUp` model (video feature upsampling).  
It mirrors the structure of `train.py` but is interactive — tweak and re-run cells freely.

**Branch:** `3Dconv`  
**Model:** `anyup.model.AnyUp` (spatiotemporal cross-attention)  
**Teacher:** VideoMAE (or any video encoder producing 5D feature tensors `(B, C, T, H, W)`)

## 0. Imports & Paths

In [ ]:
import os
import sys
import random
import datetime
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import autocast
from torch.utils.tensorboard import SummaryWriter
from tqdm.notebook import tqdm

# Make sure the repo root is on the path
REPO_ROOT = Path(".").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Repo root: {REPO_ROOT}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 1. Configuration

All knobs are here. Adjust before running the rest.

> **Colab users:** After `cfg = TrainConfig()` there is a clearly marked override block.
> That block is the only place you need to edit for each new run or new account.

In [ ]:
from dataclasses import dataclass, field
from typing import List, Optional, Tuple

@dataclass
class TStage:
    t: int           # number of frames for this stage
    start_step: int  # global step at which this stage activates
    batch_size: int  # batch size to use in this stage

@dataclass
class TrainConfig:
    # --- Experiment identity ---
    # Change this every time you start a new run or switch accounts.
    # It is used to name checkpoint folders and TensorBoard runs so you
    # can always tell which account / attempt produced which results.
    experiment_name: str = "run_01"

    # --- Paths ---
    # Defaults are local (relative) paths — fine for running outside Colab.
    # For Colab, override these in the block below to point to Drive.
    model_ckpt_2d: Optional[str] = "checkpoints/anyup_2d.pth"  # pretrained 2D weights; None = random init
    checkpoint_dir: str = "checkpoints/anyup3d"                 # overridden below for Colab
    resume: Optional[str] = None    # path to a 3D checkpoint to resume; None = fresh start
    log_dir: str = "runs/anyup3d"   # overridden below for Colab
    dataset_root: str = "data/videos"  # overridden below for Colab

    # --- Video encoder (teacher) ---
    encoder: str = "videomae"                        # only 'videomae' implemented below
    encoder_model: str = "MCG-NJU/videomae-base"     # HuggingFace model id

    # --- Model hyper-params ---
    input_dim: int = 3
    qk_dim: int = 128
    num_heads: int = 4
    window_ratio: float = 0.1
    kernel_size_lfu: int = 5
    t_k: int = 1              # temporal kernel size for 3D convs

    # --- Spatial resolution ---
    img_size: int = 224        # spatial size fed to the model
    crop_size: int = 448       # high-res crop for GT token extraction

    # --- T-curriculum (frame count ramps up over training) ---
    t_schedule: List[TStage] = field(default_factory=lambda: [
        TStage(t=1,  start_step=0,     batch_size=8),
        TStage(t=4,  start_step=5000,  batch_size=4),
        TStage(t=8,  start_step=15000, batch_size=2),
        TStage(t=16, start_step=30000, batch_size=1),
    ])

    # --- Optimizer ---
    lr: float = 2e-4
    weight_decay: float = 1e-4
    grad_clip_max_norm: float = 1.0

    # --- Loss weights ---
    lambda_cos_mse: float = 1.0
    lambda_input_consistency: float = 0.1
    lambda_self_consistency: float = 0.1
    lambda_temporal_consistency: float = 0.2
    temporal_lambda_warmup_steps: int = 5000

    # --- Training duration ---
    max_steps: int = 50_000
    save_every_n_steps: int = 500
    val_every_n_steps: int = 1000
    log_every_n_steps: int = 50

    # --- Data ---
    num_workers: int = 4
    pin_memory: bool = True

    # --- Misc ---
    seed: int = 42
    debug: bool = False
    debug_steps: int = 10


cfg = TrainConfig()

# ============================================================
# COLAB OVERRIDES — edit this block for every new run.
#
# 1. Change experiment_name each time you start fresh or switch
#    to a different Colab account (e.g. "run_01", "run_02").
#    This labels the checkpoint folder so you always know which
#    account/attempt produced which results.
#
# 2. DRIVE_ROOT must match the folder you created (or shared via
#    shortcut) in your MyDrive.  See docs/colab_training_guide.md
#    for the shortcut setup instructions.
#
# 3. dataset_root is set to '/content/local_dataset' because
#    Cell 7a copies the dataset there from Drive before training.
#    Reading from /content/ is 3-5x faster than reading from Drive.
#    If you are using a streaming dataset (Cell 7b option 3),
#    dataset_root is unused — you can leave it as-is.
# ============================================================

DRIVE_ROOT = "/content/drive/MyDrive/anyup3d"  # <-- change to your Drive folder

cfg.experiment_name = "run_01"                 # <-- change for each new run / account
cfg.model_ckpt_2d   = f"{DRIVE_ROOT}/anyup_2d.pth"
cfg.checkpoint_dir  = f"{DRIVE_ROOT}/checkpoints/{cfg.experiment_name}"
cfg.log_dir         = f"{DRIVE_ROOT}/runs/{cfg.experiment_name}"
cfg.dataset_root    = "/content/local_dataset"  # local copy — see Cell 7a

# cfg.resume = f"{DRIVE_ROOT}/checkpoints/run_01/anyup3d_step5000.pth"  # uncomment to resume
# cfg.debug  = True   # uncomment for a 10-step smoke test

print(cfg)

## 2. Reproducibility & Device

In [ ]:
import numpy as np

torch.manual_seed(cfg.seed)
np.random.seed(cfg.seed)
random.seed(cfg.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(cfg.seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on: {device}")

## 3. Mount Drive & Logger (TensorBoard)

Drive is mounted here so that checkpoints and TensorBoard logs are written to it immediately.
The free Colab GPU session lasts ~4–5 hours — anything written only to `/content/` is lost
when the session ends.

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    print("Drive mounted at /content/drive")
except ModuleNotFoundError:
    print("Not running in Colab — Drive mount skipped.")

timestamp = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
run_log_dir = Path(cfg.log_dir) / timestamp
run_log_dir.mkdir(parents=True, exist_ok=True)

writer = SummaryWriter(log_dir=str(run_log_dir))
print(f"TensorBoard log dir: {run_log_dir}")
print("Launch TensorBoard with:")
print(f"  %load_ext tensorboard")
print(f"  %tensorboard --logdir {cfg.log_dir}")

## 4. Video Encoder (Teacher / GT features)

The teacher extracts ground-truth spatiotemporal features that AnyUp is trained to reproduce.  
Currently wired for **VideoMAE**. Swap this cell to use a different encoder.

In [ ]:
# ---- VideoMAE encoder ----
# Output shape: (B, T*H*W, C) spatial tokens -> reshaped to (B, C, T, H, W)
# For ViT-B/16 at 224px: H=W=14, so token grid is (T, 14, 14)

from transformers import VideoMAEModel, AutoFeatureExtractor

print(f"Loading teacher: {cfg.encoder_model} ...")
feature_extractor = AutoFeatureExtractor.from_pretrained(cfg.encoder_model)
encoder = VideoMAEModel.from_pretrained(cfg.encoder_model).to(device).eval()

# Freeze — we never train the teacher
for p in encoder.parameters():
    p.requires_grad_(False)

# Derive encoder output dim & patch size
ENCODER_DIM = encoder.config.hidden_size     # e.g. 768 for ViT-B
PATCH_SIZE  = encoder.config.patch_size      # e.g. 16
TOKEN_H = cfg.img_size // PATCH_SIZE
TOKEN_W = cfg.img_size // PATCH_SIZE

print(f"Encoder dim={ENCODER_DIM}, patch_size={PATCH_SIZE}, spatial token grid={TOKEN_H}x{TOKEN_W}")

## 5. AnyUp 3D Model

In [ ]:
from anyup.model import AnyUp

anyup = AnyUp(
    input_dim=cfg.input_dim,
    qk_dim=cfg.qk_dim,
    num_heads=cfg.num_heads,
    window_ratio=cfg.window_ratio,
    kernel_size_lfu=cfg.kernel_size_lfu,
    t_k=cfg.t_k,
).to(device)

# --- Optional: load 2D pretrained weights ---
if cfg.model_ckpt_2d and Path(cfg.model_ckpt_2d).exists():
    print(f"Loading 2D checkpoint: {cfg.model_ckpt_2d}")
    state = torch.load(cfg.model_ckpt_2d, map_location="cpu", weights_only=False)
    sd = state.get("anyup", state)
    sd = sd.get("state_dict", sd)
    missing, unexpected = anyup.load_state_dict(sd, strict=False)
    print(f"  Missing keys : {len(missing)}")
    print(f"  Unexpected   : {len(unexpected)}")
elif cfg.model_ckpt_2d:
    print(f"[WARNING] 2D checkpoint not found at '{cfg.model_ckpt_2d}' — starting from random init")
else:
    print("Starting from random init (no 2D checkpoint configured)")

# --- Optional: resume 3D checkpoint ---
global_step = 0
data_epoch  = 0   # tracks how many full passes through the dataset we've done;
                  # used to seed the DataLoader so each pass has a different shuffle order
if cfg.resume and Path(cfg.resume).exists():
    print(f"Resuming from: {cfg.resume}")
    ckpt = torch.load(cfg.resume, map_location="cpu", weights_only=False)
    anyup.load_state_dict(ckpt["anyup"])
    global_step = ckpt.get("global_step", 0)
    data_epoch  = ckpt.get("data_epoch",  0)  # restore shuffle position
    print(f"  Resumed at step {global_step}, data_epoch {data_epoch}")

anyup.train()
num_params = sum(p.numel() for p in anyup.parameters() if p.requires_grad)
print(f"AnyUp 3D trainable params: {num_params:,}")

## 6. Loss Functions

In [ ]:
from anyup.loss import Cosine_MSE

criterion = Cosine_MSE()

def cosine_mse_3d(pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    """
    pred, target: (B, C, T, H, W)  ->  flatten T,H,W then call 2D criterion.
    Returns scalar loss.
    """
    B, C, T, H, W = pred.shape
    pred_2d   = pred.view(B, C, T * H, W)
    target_2d = target.view(B, C, T * H, W)
    return criterion(pred_2d, target_2d)["total"]


def get_temporal_lambda(step: int) -> float:
    """Linearly ramp temporal consistency weight from 0 to cfg value."""
    warmup = cfg.temporal_lambda_warmup_steps
    if warmup <= 0:
        return cfg.lambda_temporal_consistency
    return cfg.lambda_temporal_consistency * min(1.0, step / warmup)


print("Loss functions ready.")

## 7a. Dataset Setup — Copy from Drive to Local Storage

Colab accesses Drive through a FUSE filesystem which adds latency on every file read.
For video datasets this noticeably slows down the DataLoader.  
**Copying (or unzipping) the dataset to `/content/` first is 3–5x faster** because
`/content/` is backed by the instance's local disk.

**Skip this cell** if you are using a streaming dataset (Option 3 in Cell 7b) — streaming
downloads directly from the internet and does not use `cfg.dataset_root` at all.

In [ ]:
import shutil
import zipfile

# ---- Configure these two paths ----
DRIVE_DATASET_SOURCE = "/content/drive/MyDrive/anyup3d/datasets/videos.zip"  # zip OR folder in Drive
LOCAL_DATASET_PATH   = "/content/local_dataset"                               # destination on local disk
# -----------------------------------

src = Path(DRIVE_DATASET_SOURCE)
dst = Path(LOCAL_DATASET_PATH)

if dst.exists():
    print(f"Local dataset already exists at {dst} — skipping copy.")
    print("Delete it manually if you want a fresh copy:  !rm -rf /content/local_dataset")

elif not src.exists():
    print(f"[WARNING] Source not found: {src}")
    print("Keeping cfg.dataset_root as-is — make sure it points to valid data before training.")

elif src.suffix == ".zip":
    print(f"Unzipping {src} -> {dst} ...")
    dst.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(src, "r") as zf:
        zf.extractall(dst)
    print(f"Done. Contents: {list(dst.iterdir())[:5]}")

else:
    print(f"Copying folder {src} -> {dst} ...")
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print(f"Done. Contents: {list(dst.iterdir())[:5]}")

cfg.dataset_root = LOCAL_DATASET_PATH
print(f"cfg.dataset_root = {cfg.dataset_root}")

## 7b. Dataset & DataLoader

Pick **one** of the three options below. Comment out the other two.

| Option | Drive space needed | Download | Good for |
|--------|-------------------|----------|----------|
| **1 — UCF-101** | 0 (downloads to `/content/`) | ~7 GB once | Real training, no Drive space |
| **2 — HMDB-51** | 0 (downloads to `/content/`) | ~2 GB once | Quick experiments, no Drive space |
| **3 — Streaming (Kinetics)** | 0 (streamed on-the-fly) | None | Large-scale training, no storage at all |
| **4 — Stub (random tensors)** | 0 | None | Smoke test only — verifies shapes, not real training |

The `build_loader(t, batch_size, epoch)` function must be defined no matter which option
you use — the training loop calls it with these three arguments.

In [ ]:
from torch.utils.data import DataLoader, Dataset, IterableDataset
import torchvision.transforms.v2 as Tv2
from torchvision.transforms import InterpolationMode

# ----------------------------------------------------------------
# Shared frame transform — applied the same way in all options
# ----------------------------------------------------------------
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

frame_tf = Tv2.Compose([
    Tv2.ToImage(),
    Tv2.Resize(cfg.img_size, interpolation=InterpolationMode.BILINEAR, antialias=True),
    Tv2.CenterCrop(cfg.img_size),
    Tv2.ToDtype(torch.float32, scale=True),
    Tv2.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# ================================================================
# OPTION 1 — UCF-101  (~7 GB, downloads to /content/)
# torchvision.datasets.UCF101 requires a separate annotation zip.
# Run the two !wget lines once, then this cell handles the rest.
# ================================================================

# # Step 1 — download data & annotations (run once in a separate cell):
# !wget -q https://www.crcv.ucf.edu/data/UCF101/UCF101.rar -P /content/ucf101/
# !wget -q https://www.crcv.ucf.edu/data/UCF101/UCF101TrainTestSplits-RecognitionTask.zip -P /content/ucf101/
# !unrar x /content/ucf101/UCF101.rar /content/ucf101/ -y
# !unzip -q /content/ucf101/UCF101TrainTestSplits-RecognitionTask.zip -d /content/ucf101/

# import torchvision

# class UCF101VideoDataset(Dataset):
#     """
#     Wraps torchvision UCF101 to return our expected dict format.
#     Each item: video tensor (T, C, H, W), ImageNet-normalised.
#     """
#     def __init__(self, root, annotation_path, frames_per_clip, split="train"):
#         self.inner = torchvision.datasets.UCF101(
#             root=root,
#             annotation_path=annotation_path,
#             frames_per_clip=frames_per_clip,
#             step_between_clips=4,
#             fold=1,
#             train=(split == "train"),
#             transform=None,
#         )

#     def __len__(self):
#         return len(self.inner)

#     def __getitem__(self, idx):
#         video, audio, label = self.inner[idx]
#         # video: (T, H, W, C) uint8 -> apply frame_tf per frame
#         frames = torch.stack([frame_tf(video[i]) for i in range(video.shape[0])])
#         return {"video": frames}  # (T, C, H, W)

# def build_loader(t: int, batch_size: int, split: str = "train", epoch: int = 0) -> DataLoader:
#     g = torch.Generator()
#     g.manual_seed(cfg.seed + epoch)   # different shuffle every pass through the data
#     dataset = UCF101VideoDataset(
#         root="/content/ucf101/UCF-101",
#         annotation_path="/content/ucf101/ucfTrainTestlist",
#         frames_per_clip=t,
#         split=split,
#     )
#     return DataLoader(
#         dataset, batch_size=batch_size, shuffle=(split == "train"),
#         generator=g, num_workers=cfg.num_workers if not cfg.debug else 0,
#         pin_memory=cfg.pin_memory and device.type == "cuda", drop_last=True,
#     )


# ================================================================
# OPTION 2 — HMDB-51  (~2 GB, downloads to /content/)
# ================================================================

# # Step 1 — download (run once):
# !wget -q http://serre-lab.clps.brown.edu/wp-content/uploads/2013/10/hmdb51_org.rar -P /content/hmdb51/
# !wget -q http://serre-lab.clps.brown.edu/wp-content/uploads/2013/10/test_train_splits.rar -P /content/hmdb51/
# !unrar x /content/hmdb51/hmdb51_org.rar /content/hmdb51/ -y
# !unrar x /content/hmdb51/test_train_splits.rar /content/hmdb51/ -y

# import torchvision

# class HMDB51VideoDataset(Dataset):
#     def __init__(self, root, annotation_path, frames_per_clip, split="train"):
#         self.inner = torchvision.datasets.HMDB51(
#             root=root,
#             annotation_path=annotation_path,
#             frames_per_clip=frames_per_clip,
#             step_between_clips=4,
#             fold=1,
#             train=(split == "train"),
#             transform=None,
#         )

#     def __len__(self):
#         return len(self.inner)

#     def __getitem__(self, idx):
#         video, audio, label = self.inner[idx]
#         frames = torch.stack([frame_tf(video[i]) for i in range(video.shape[0])])
#         return {"video": frames}

# def build_loader(t: int, batch_size: int, split: str = "train", epoch: int = 0) -> DataLoader:
#     g = torch.Generator()
#     g.manual_seed(cfg.seed + epoch)
#     dataset = HMDB51VideoDataset(
#         root="/content/hmdb51/hmdb51_org",
#         annotation_path="/content/hmdb51/testTrainMulti_7030_splits",
#         frames_per_clip=t,
#         split=split,
#     )
#     return DataLoader(
#         dataset, batch_size=batch_size, shuffle=(split == "train"),
#         generator=g, num_workers=cfg.num_workers if not cfg.debug else 0,
#         pin_memory=cfg.pin_memory and device.type == "cuda", drop_last=True,
#     )


# ================================================================
# OPTION 3 — Kinetics via HuggingFace streaming
# No storage needed at all — data is streamed batch-by-batch.
# Requires: pip install datasets av
# ================================================================

# import av
# from datasets import load_dataset
# import io

# def decode_video_bytes(video_bytes: bytes, num_frames: int, img_size: int):
#     """Decode raw video bytes and sample num_frames evenly."""
#     container = av.open(io.BytesIO(video_bytes))
#     stream = container.streams.video[0]
#     total = stream.frames
#     indices = torch.linspace(0, total - 1, num_frames).long().tolist()
#     frames = []
#     for i, frame in enumerate(container.decode(video=0)):
#         if i in indices:
#             img = frame.to_image().convert("RGB").resize((img_size, img_size))
#             frames.append(frame_tf(img))
#         if len(frames) == num_frames:
#             break
#     container.close()
#     # Pad with last frame if stream ended early
#     while len(frames) < num_frames:
#         frames.append(frames[-1])
#     return torch.stack(frames)  # (T, C, H, W)


# class StreamingKineticsDataset(IterableDataset):
#     """
#     Wraps the HuggingFace Kinetics-400 streaming split.
#     Yields dicts with key 'video': (T, C, H, W).
#     """
#     def __init__(self, num_frames: int, img_size: int, split: str = "train"):
#         self.num_frames = num_frames
#         self.img_size   = img_size
#         # streaming=True means no files are downloaded — each item is fetched on-the-fly
#         self.hf_dataset = load_dataset(
#             "HuggingFaceM4/kinetics400", split=split, streaming=True, trust_remote_code=True
#         )

#     def __iter__(self):
#         for item in self.hf_dataset:
#             try:
#                 video = decode_video_bytes(
#                     item["video"]["bytes"],
#                     num_frames=self.num_frames,
#                     img_size=self.img_size,
#                 )
#                 yield {"video": video}
#             except Exception:
#                 continue  # skip corrupted clips silently

# def build_loader(t: int, batch_size: int, split: str = "train", epoch: int = 0) -> DataLoader:
#     # IterableDataset with streaming: shuffle happens server-side, epoch arg unused here
#     dataset = StreamingKineticsDataset(num_frames=t, img_size=cfg.img_size, split=split)
#     return DataLoader(
#         dataset, batch_size=batch_size,
#         num_workers=0,  # must be 0 for IterableDataset in Colab
#         pin_memory=cfg.pin_memory and device.type == "cuda",
#     )


# ================================================================
# OPTION 4 — Stub (random tensors, smoke test only)
# ================================================================

class StubVideoDataset(Dataset):
    """Generates random tensors. Replace with a real dataset above."""
    def __init__(self, num_samples=1000, t=1, img_size=224):
        self.num_samples = num_samples
        self.t = t
        self.img_size = img_size

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        video = torch.randn(self.t, 3, self.img_size, self.img_size)
        return {"video": video, "augmented_video": video + 0.05 * torch.randn_like(video)}


def build_loader(t: int, batch_size: int, split: str = "train", epoch: int = 0) -> DataLoader:
    # epoch is used to seed the generator so each pass through the data
    # produces a DIFFERENT shuffle order — important for correct resumability.
    g = torch.Generator()
    g.manual_seed(cfg.seed + epoch)
    dataset = StubVideoDataset(
        num_samples=500 if cfg.debug else 50_000,
        t=t,
        img_size=cfg.img_size,
    )
    return DataLoader(
        dataset, batch_size=batch_size, shuffle=(split == "train"),
        generator=g, num_workers=cfg.num_workers if not cfg.debug else 0,
        pin_memory=cfg.pin_memory and device.type == "cuda", drop_last=True,
    )


# Quick sanity check
test_loader = build_loader(t=1, batch_size=2, epoch=0)
sample_batch = next(iter(test_loader))
print("Sample batch shapes:")
for k, v in sample_batch.items():
    print(f"  {k}: {v.shape}")

## 8. Optimizer

In [ ]:
optimizer = torch.optim.AdamW(
    anyup.parameters(),
    lr=cfg.lr,
    weight_decay=cfg.weight_decay,
)

# Cosine LR decay over the full training run
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=cfg.max_steps,
    eta_min=cfg.lr * 0.01,
)

if cfg.resume and Path(cfg.resume).exists():
    ckpt = torch.load(cfg.resume, map_location="cpu", weights_only=False)
    if "optimizer" in ckpt:
        optimizer.load_state_dict(ckpt["optimizer"])
    if "scheduler" in ckpt:
        scheduler.load_state_dict(ckpt["scheduler"])

print("Optimizer and scheduler ready.")

## 9. Checkpoint Helpers

In [ ]:
ckpt_dir = Path(cfg.checkpoint_dir)
ckpt_dir.mkdir(parents=True, exist_ok=True)

def save_checkpoint(step: int, data_epoch: int, tag: str = ""):
    """
    data_epoch: how many full passes through the dataset have been completed.
    Saved in the checkpoint so that on resume the DataLoader resumes with
    the correct (next) shuffle order and does not replay already-seen data.
    """
    name = f"anyup3d_step{step}{('_' + tag) if tag else ''}.pth"
    path = ckpt_dir / name
    torch.save({
        "anyup": anyup.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "global_step": step,
        "data_epoch":  data_epoch,
        "experiment_name": cfg.experiment_name,
        "cfg": vars(cfg),
    }, path)
    print(f"Saved checkpoint: {path}")
    return path

print(f"Experiment : {cfg.experiment_name}")
print(f"Checkpoints: {ckpt_dir}")

## 10. Training Loop

The loop implements:
1. **T-curriculum** — batch size and frame count change at milestone steps
2. **Primary loss** — `Cosine_MSE` between AnyUp output and teacher GT features
3. **Input-consistency** — same video, different resolution → features should match
4. **Self-consistency** — augmented video → features should be close to clean
5. **Temporal consistency** — adjacent frames → features should be smooth (ramped in)

**Data ordering across sessions:** The DataLoader is seeded with `cfg.seed + data_epoch`.
Each time the iterator runs out of data (one full pass through the dataset), `data_epoch`
increments and the next pass gets a different shuffle seed. This value is saved in every
checkpoint and restored on resume, so after reconnecting the model always continues
into data it has not seen in the same shuffle order.

In [ ]:
# ---- Utility: extract teacher features from a video batch ----
@torch.no_grad()
def extract_teacher_features(video: torch.Tensor) -> torch.Tensor:
    """
    video : (B, T, C, H, W) float32, ImageNet-normalized
    returns: (B, C_enc, T, H_tok, W_tok)  — 5D feature volume
    """
    B, T, C, H, W = video.shape
    pixel_values = video.permute(0, 2, 1, 3, 4).to(device)  # (B, C, T, H, W)
    out = encoder(pixel_values=pixel_values)
    tokens = out.last_hidden_state  # (B, N_tokens, C_enc)
    N = tokens.shape[1]
    H_tok = W_tok = int((N / T) ** 0.5)
    assert H_tok * W_tok * T == N, f"Cannot reshape {N} tokens into ({T}, {H_tok}, {W_tok})"
    feats = tokens.reshape(B, T, H_tok, W_tok, -1).permute(0, 4, 1, 2, 3).contiguous()
    return feats


# ---- Utility: determine current curriculum stage ----
def get_current_stage(step: int) -> TStage:
    stage = cfg.t_schedule[0]
    for s in cfg.t_schedule:
        if step >= s.start_step:
            stage = s
    return stage


print("Training utilities ready.")

In [ ]:
# ---- Main training loop ----

max_steps = cfg.debug_steps if cfg.debug else cfg.max_steps
current_stage = get_current_stage(global_step)

# data_epoch was restored from checkpoint in Cell 5 (or is 0 for a fresh run).
# Pass it to build_loader so the shuffle seed is correct from the start.
loader     = build_loader(t=current_stage.t, batch_size=current_stage.batch_size, epoch=data_epoch)
loader_iter = iter(loader)

anyup.train()

pbar = tqdm(range(global_step, max_steps), initial=global_step, total=max_steps, desc="Training")

for step in pbar:

    # ---- Check if curriculum stage changed ----
    new_stage = get_current_stage(step)
    if new_stage.t != current_stage.t or new_stage.batch_size != current_stage.batch_size:
        print(f"\n[Step {step}] Curriculum: T={current_stage.t}->{new_stage.t}, "
              f"BS={current_stage.batch_size}->{new_stage.batch_size}")
        current_stage = new_stage
        # Stage change resets the loader; bump data_epoch so the new pass is
        # independently shuffled from what was seen before.
        data_epoch += 1
        loader     = build_loader(t=current_stage.t, batch_size=current_stage.batch_size, epoch=data_epoch)
        loader_iter = iter(loader)

    # ---- Get batch (reset iterator when the dataset is exhausted) ----
    try:
        batch = next(loader_iter)
    except StopIteration:
        # One full pass through the data is done. Increment data_epoch so the
        # next pass is seeded differently — this is the key to not replaying
        # the same shuffle order after a session disconnect and resume.
        data_epoch += 1
        loader     = build_loader(t=current_stage.t, batch_size=current_stage.batch_size, epoch=data_epoch)
        loader_iter = iter(loader)
        batch      = next(loader_iter)

    video     = batch["video"].to(device)           # (B, T, C, H, W)
    aug_video = batch.get("augmented_video")
    if aug_video is not None:
        aug_video = aug_video.to(device)

    B, T, C, H, W = video.shape

    # ---- Forward pass ----
    with autocast(device_type=device.type, dtype=torch.bfloat16):

        gt_feats  = extract_teacher_features(video)          # (B, C_enc, T, H_tok, W_tok)
        video_5d  = video.permute(0, 2, 1, 3, 4)            # (B, C, T, H, W)
        out_size  = (gt_feats.shape[-2], gt_feats.shape[-1]) # (H_tok, W_tok)
        pred_feats = anyup(video_5d, gt_feats, output_size=out_size)

        # 1. Primary reconstruction loss
        loss_rec = cosine_mse_3d(pred_feats, gt_feats) * cfg.lambda_cos_mse

        # 2. Self-consistency under augmentation
        loss_aug = torch.tensor(0.0, device=device)
        if aug_video is not None and cfg.lambda_self_consistency > 0:
            aug_5d = aug_video.permute(0, 2, 1, 3, 4)
            with torch.no_grad():
                gt_feats_aug = extract_teacher_features(aug_video)
            pred_aug = anyup(aug_5d, gt_feats_aug, output_size=out_size)
            loss_aug = cosine_mse_3d(pred_aug, pred_feats.detach()) * cfg.lambda_self_consistency

        # 3. Temporal consistency
        loss_temp = torch.tensor(0.0, device=device)
        t_lam = get_temporal_lambda(step)
        if T > 1 and t_lam > 0:
            diff = pred_feats[:, :, 1:] - pred_feats[:, :, :-1]
            loss_temp = diff.pow(2).mean() * t_lam

        total_loss = loss_rec + loss_aug + loss_temp

    # ---- Backward ----
    optimizer.zero_grad()
    total_loss.backward()
    torch.nn.utils.clip_grad_norm_(anyup.parameters(), cfg.grad_clip_max_norm)
    optimizer.step()
    scheduler.step()

    # ---- Logging ----
    if step % cfg.log_every_n_steps == 0:
        lr = optimizer.param_groups[0]["lr"]
        writer.add_scalar("loss/total",    total_loss.item(), step)
        writer.add_scalar("loss/rec",      loss_rec.item(),   step)
        writer.add_scalar("loss/aug",      loss_aug.item(),   step)
        writer.add_scalar("loss/temporal", loss_temp.item(),  step)
        writer.add_scalar("lr",            lr,                step)
        writer.add_scalar("curriculum/T",  current_stage.t,   step)
        writer.add_scalar("curriculum/bs", current_stage.batch_size, step)
        writer.add_scalar("data/epoch",    data_epoch,        step)

        pbar.set_postfix({
            "loss":       f"{total_loss.item():.4f}",
            "rec":        f"{loss_rec.item():.4f}",
            "T":          current_stage.t,
            "lr":         f"{lr:.2e}",
            "data_epoch": data_epoch,
        })

    # ---- Checkpoint ----
    if (step + 1) % cfg.save_every_n_steps == 0:
        save_checkpoint(step + 1, data_epoch)

    if cfg.debug and step >= cfg.debug_steps - 1:
        print(f"Debug mode: stopping after {cfg.debug_steps} steps.")
        break


# Final checkpoint
save_checkpoint(max_steps, data_epoch, tag="final")
writer.flush()
writer.close()
print("Training complete.")

## 11. Quick Validation / Smoke Test

Run this at any point to check the model produces sensible outputs without running the full training loop.

In [ ]:
anyup.eval()

with torch.no_grad():
    dummy_video = torch.randn(1, 4, 3, cfg.img_size, cfg.img_size, device=device)
    dummy_feats = torch.randn(1, ENCODER_DIM, 4, TOKEN_H, TOKEN_W, device=device)
    dummy_video_5d = dummy_video.permute(0, 2, 1, 3, 4)

    out = anyup(dummy_video_5d, dummy_feats, output_size=(TOKEN_H, TOKEN_W))
    print(f"Input  features : {dummy_feats.shape}")
    print(f"Output features : {out.shape}")
    assert out.shape == dummy_feats.shape, "Shape mismatch!"
    print("Validation passed.")

anyup.train()